In [2]:
#importing relevant libraries
import pandas as pd
pd.options.mode.chained_assignment = None
import pylab as pl
import numpy as np
import scipy.optimize as opt
from sklearn import preprocessing
%matplotlib inline 
import matplotlib.pyplot as plt

In [3]:
#Reading the file
data = pd.read_csv('plan b data.csv')
#Displaying the first 5 rows
data.head()

,Patients,Diagnosis,VisualG,SUVmax,TBR,ConclusionPET,ABusepostoper,minimum_intensity,maximum_intensity,range,...,gray_level_nonuniformity_A,size_zone_nonuniformity,zone_percentage,low_gray_level_zone_emphasis,high_gray_level_zone_emphasis,small_zone_low_gray_level_emphasis,small_zone_high_gray_level_emphasis,large_zone_low_gray_level_emphasis,large_zone_high_gray_level_emphasis,filter_$
0,1,1,4,6.7,3.72,2,1,0.62,6.68,6.07,...,16.08,25.22,0.0,0.23,9.939963e+07,0.01,9.860538e+07,11.75,1.229531e+08,0
1,2,1,3,9.8,4.90,1,1,0.34,9.66,9.32,...,68.59,37.58,0.0,0.40,2.941573e+08,0.01,2.939368e+08,28.57,3.034294e+08,0
2,3,1,3,5.3,4.42,1,1,0.08,5.28,5.20,...,20.20,18.46,0.0,0.36,1.688961e+09,0.03,1.688835e+09,10.02,1.691300e+09,0
3,4,1,4,11.2,10.18,1,1,0.47,8.50,8.03,...,11.73,8.39,0.0,0.29,1.263940e+07,0.01,1.259057e+07,31.82,1.366822e+07,0
4,5,1,4,9.1,6.59,1,1,0.60,8.30,7.70,...,10.32,5.95,0.0,0.36,1.278671e+07,0.01,1.277919e+07,31.71,1.305688e+07,0


In [4]:
#Size of the data
data.shape

(30, 108)

In [5]:
#In practice, feature selection should be done after data pre-processing, so ideally, all the categorical variables are encoded into numbers
numerics = ['int16','int32','int64','float16','float32','float64']
numerical_vars = list(data.select_dtypes(include=numerics).columns)
data = data[numerical_vars]

In [6]:
#Checking missing values in the data
data.isnull().sum()

Patients                               0
Diagnosis                              0
VisualG                                0
SUVmax                                 0
TBR                                    0
                                      ..
small_zone_low_gray_level_emphasis     0
small_zone_high_gray_level_emphasis    0
large_zone_low_gray_level_emphasis     0
large_zone_high_gray_level_emphasis    0
filter_$                               0
Length: 108, dtype: int64

In [7]:
#Creating the training data
X = data.drop(['Diagnosis'], axis=1)
y = data['Diagnosis']

In [8]:
X.shape, y.shape

((30, 107), (30,))

In [9]:
#Importing the models
from mlxtend.feature_selection import SequentialFeatureSelector as sfs
from sklearn.linear_model import LinearRegression

In [10]:
#Calling the linear regression model
lreg = LinearRegression()
sfs1 = sfs(lreg, k_features=10, forward=True, verbose=2, scoring='neg_mean_squared_error')

In [11]:
sfs1 = sfs1.fit(X, y)

[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done 107 out of 107 | elapsed:    1.3s finished

[2022-03-11 14:21:35] Features: 1/10 -- score: -0.14692480795043145[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done 106 out of 106 | elapsed:    1.1s finished

[2022-03-11 14:21:37] Features: 2/10 -- score: -0.06533191411438635[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    0.0s remaining:    0.0s
[Parallel(n_jobs=1)]: Done 105 out of 105 | elapsed:    1.0s finished

[2022-03-11 14:21:38] Features: 3/10 -- score: -0.04613470498494304[Parallel(n_jobs=1)]: Using backend SequentialBackend with 1 concurrent workers.
[Parallel(n_jobs=1)]: 

In [12]:
#Displaying the list of the selected features
feat_names = list(sfs1.k_feature_names_)
print(feat_names)

['Patients', 'VisualG', 'ConclusionPET', 'quantile0025', 'quantile25', 'kurtosis', 'information_measure_of_correlation1_max', 'long_run_low_gray_level_emphasis_mean', 'long_run_high_gray_level_emphasis_max', 'filter_$']


In [13]:
# creating a new dataframe using the above variables and adding the target variable
new_data = data[feat_names]
new_data['Diagnosis'] = data['Diagnosis']

# first five rows of the new data
new_data.head()

,Patients,VisualG,ConclusionPET,quantile0025,quantile25,kurtosis,information_measure_of_correlation1_max,long_run_low_gray_level_emphasis_mean,long_run_high_gray_level_emphasis_max,filter_$,Diagnosis
0,1,4,2,1.13,1.77,4.20,0.67,17.75,954.69,0,1
1,2,3,1,0.96,1.90,5.22,0.67,18.01,2014.06,0,1
2,3,3,1,0.32,1.00,2.58,0.74,6.02,13364.38,0,1
3,4,4,1,0.78,1.29,4.91,0.67,19.60,668.05,0,1
4,5,4,1,0.99,1.70,3.51,0.62,19.58,693.95,0,1


In [14]:
# shape of new and original data
new_data.shape, data.shape

((30, 11), (30, 108))